# Tutorial 9: Multi-reference to recover locations in mouse brain scRNA-seq

In [1]:
import remap
import anndata as ad
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

## Load ST reference and scRNA-seq data
##### Source data can be downloaded from the link: https://drive.google.com/drive/folders/1_0zQ4MP8-Aol55u-ghfhSVslbpq4QPns
##### For multi-capture training, we combined all ST captures into one anndata object, with a column in the adata.obs to indicate the source labels. In this data, ```Rdata.obs['source']``` indicates the capture ID.
##### The batch effect corrected expressions are stored in `Rdata.obsm['corrected']` and `Qdata.obsm['corrected']`. If no external batch correction is applied, setting `harmony = TRUE` in the `remap.Fit_cord_multi` function will run Harmony internally to remove batch effects.

In [2]:
Rdata = ad.read_h5ad("data/brain_sc/st_data.h5ad")
Qdata = ad.read_h5ad("data/brain_sc/sc_data.h5ad")
print(Rdata)
print(Qdata)

/home/shunzhou/miniconda3/envs/py39/lib/python3.9/site-packages/anndata/_core/anndata.py:1754: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


AnnData object with n_obs × n_vars = 213472 × 248
    obs: 'x_cord', 'y_cord', 'n_genes', 'n_counts', 'Cluster', 'source'
    var: 'gene_ids', 'feature_types', 'genome'
    obsm: 'spatial'
AnnData object with n_obs × n_vars = 24832 × 21887
    obs: 'Cluster', 'source', 'donor_sex_id', 'donor_sex_label'
    var: 'n_cells'
    uns: 'downsampling'
    obsm: 'covet'


## Initialize neighboring gene-gene covariance estimation.
##### For multi-capture training, please add the parameter `batch_key = 'source'` to indicate the capture source label column.
##### After training, initialized neighboring gene-gene covariance for ST and scRNA-seq will be saved to `path_name` in `npy` format, named as `st_covariance.npy` and `sc_covariance.npy`.

In [3]:
data_name = "brain_sc"
path_name =  f"remap_output/{data_name}"

Rdata, Qdata = remap.covet_init(st_data = Rdata, sc_data = Qdata, num_covet_genes=100, k_nearest=100, num_HVG=1000, batch_key = 'source',
                 save_path = path_name, covet_batch_size = None, log_input = 0.0, lib_size = False)

/home/shunzhou/miniconda3/envs/py39/lib/python3.9/site-packages/anndata/_core/anndata.py:1754: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/shunzhou/miniconda3/envs/py39/lib/python3.9/site-packages/anndata/_core/anndata.py:1754: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/shunzhou/miniconda3/envs/py39/lib/python3.9/site-packages/anndata/_core/anndata.py:1754: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Number of genes used for calculating neighboring gene-gene covariance: 100
Training ENVI on cuda


Trn: spatial Loss: -0.77921, SC Loss: -0.40801, Cov Loss: -0.00078, KL Loss: 0.65737: 100%|██████████████████████████████████████████████████████████████| 10000/10000 [03:22<00:00, 49.43it/s]


Finished initializing neighboring gene-gene covariance estimation.
Saved neighboring gene-gene covariance PCA embeddings.


## Fit location prediction model.
##### We use `Fit_cord_multi` to train the prediction model with multiple ST captures. `equal_size` is a parameter to indicate whether each ST capture has equal size. If not, please change `equal_size=False`, and the locations of each ST capture will be rescaled by their location range.
##### For multi-capture training, please add the parameter `batch_key = 'source'` to indicate the capture source label column.
##### In this notebook the goal is to reconstruct the 2D embedding of the query cells, which requires the distance between every pair of cells. We therefore set `full_pairwise = True`, so that the complete `n x n` pairwise distance matrix is predicted. (If the primary goal were instead to identify spatial neighbors and construct CN clusters/spatial niches, the full matrix would be unnecessary: setting `full_pairwise = False` predicts distances only for candidate neighbors of each cell, which is much faster and lighter.)
##### The predicted pairwise distance will be saved to `path_name` in `npz` format.
##### If no external batch effect correction methods are applied, setting `harmony = True` is recommended to remove gene expression batch effect using Harmony

In [4]:
dist_pred = remap.Fit_cord_multi(Rdata = Rdata, location_data = Rdata.obs[['x_cord', 'y_cord']], Qdata = Qdata, path_name = path_name, 
                                 full_pairwise = True,
                                 source_key = "source", equal_size = True, harmony = True)
print(dist_pred.shape)

Model is on GPU


/home/shunzhou/miniconda3/envs/py39/lib/python3.9/site-packages/anndata/_core/anndata.py:1754: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/shunzhou/miniconda3/envs/py39/lib/python3.9/site-packages/anndata/_core/anndata.py:1754: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/shunzhou/miniconda3/envs/py39/lib/python3.9/site-packages/anndata/_core/anndata.py:1754: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/shunzhou/miniconda3/envs/py39/lib/python3.9/site-packages/anndata/_core/anndata.py:1754: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


(238304, 231)


2026-08-29 13:55:00,687 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
2026-08-29 13:59:17,977 - harmonypy - INFO - sklearn.KMeans initialization complete.
2026-08-29 13:59:19,769 - harmonypy - INFO - Iteration 1 of 10
2026-08-29 14:01:08,560 - harmonypy - INFO - Iteration 2 of 10
2026-08-29 14:02:22,372 - harmonypy - INFO - Converged after 2 iterations
/home/shunzhou/miniconda3/envs/py39/lib/python3.9/site-packages/anndata/_core/anndata.py:1754: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/shunzhou/miniconda3/envs/py39/lib/python3.9/site-packages/anndata/_core/anndata.py:1754: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/shunzhou/miniconda3/envs/py39/lib/python3.9/site-packages/umap/distances.py:1063: NumbaDeprecationWarning: The 'nopython' keyword argument was not supp

Early stopping criteria reached.
Predicting pairwise distances for scRNA-seq data.


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 37638/37638 [44:23<00:00, 14.13it/s]


(24832, 24832)


##### The predicted distances are written to `{path_name}/remap_rel_dist.npz` and can be reloaded with `load_npz`, so the training and prediction steps above do not have to be re-run.
##### With `full_pairwise = True` every entry of the matrix is predicted, so the saved file is only nominally sparse (all `n x n` entries are stored). `remap.MDSTransform` symmetrizes the matrix and masks its diagonal with dense NumPy operations, and `sklearn.manifold.MDS` does not accept sparse input, so the matrix has to be converted with `.toarray()` before it is passed on.

In [5]:
from scipy.sparse import load_npz

dist_full = load_npz(f"{path_name}/remap_rel_dist.npz")
dist_full = dist_full.toarray()

## Multi-dimensional scaling (MDS) to recover the 2D embeddings

#### `remap.MDSTransform` applies MDS to the predicted pairwise distance matrix and returns a 2D embedding of the query cells, recovering their spatial organization without using any ground-truth coordinates. The embedding is stored in `Qdata.obsm['remap']` and can be plotted like any other embedding; note that MDS is defined only up to rotation, reflection and scaling, so the recovered map may appear rotated or mirrored relative to the true tissue (`remap.MDSalign` can align it to true coordinates by Procrustes analysis when they are available, for visualization only).
#### Because the reconstruction relies on the distance between every pair of cells, the full pairwise distance matrix is required here, i.e. `full_pairwise = True` above; a neighbor-filtered matrix would leave most pairs undefined and cannot be used for MDS.

In [9]:
mds_emb = remap.MDSTransform(dist_full)
np.save(f"{path_name}/mds_emb.npy", mds_emb)

mds_emb = np.load(f"{path_name}/mds_emb.npy", allow_pickle = True)

Qdata.obsm['remap'] = mds_emb
Qdata.obsm['remap']

array([[-0.22322172,  0.27188417],
       [-0.03305233,  0.15896309],
       [ 0.49775742,  0.21973736],
       ...,
       [ 0.05745324, -0.37074657],
       [ 0.43684319,  0.17255933],
       [ 0.47472179,  0.34320878]])